# Conteo de rocas: watershed (clásico) vs. FastSAM (local)

Alternativa **local** a Amazon Rekognition, usando **FastSAM** (Segment Anything rápido)
pre-entrenado, sobre tu **MacBook M4 Pro** (GPU vía `mps`). No requiere AWS ni entrenamiento.

FastSAM segmenta la imagen de forma *class-agnostic*; para contar **rocas** de forma
comparable al watershed, nos quedamos con las máscaras que caen dentro de la región
etiquetada como **big rock** (clase 3) de AI4Mars.

**Requisito:** `pip install ultralytics` en tu entorno conda `tesis-marte`.


## 1. Configuración e importaciones


In [ ]:
import sys
from pathlib import Path

# Localizar la raíz del repo (donde está la carpeta src/) para poder importar.
here = Path.cwd()
root = next((p for p in [here, *here.parents] if (p / "src").is_dir()), here)
sys.path.insert(0, str(root))

import numpy as np
import pandas as pd
import torch

from src import config, mask_utils as mu, rock_count as rc, sam_compare as sc

DEVICE = "mps" if torch.backends.mps.is_available() else "cpu"
print("Dispositivo:", DEVICE)

RESULTS = pd.read_csv(root / "outputs" / "results.csv")
model = sc.load_fastsam("FastSAM-s.pt")   # descarga pesos la primera vez
print("FastSAM cargado.")

## 2. Elegir imágenes con big rock

Tomamos una muestra de imágenes `ok` (con roca grande) variando el nº de rocas del watershed.


In [ ]:
N = 12  # nº de imágenes a comparar (súbelo si quieres)
ok = RESULTS[RESULTS.quality_flag == "ok"].copy()
sample = (ok.sort_values("n_rocks")
            .iloc[np.linspace(0, len(ok) - 1, N).astype(int)])   # variedad de conteos
ids = sample.image_id.tolist()
print(f"{len(ids)} imágenes seleccionadas (n_rocks watershed: "
      f"{sample.n_rocks.min()}–{sample.n_rocks.max()})")

## 3. Comparar watershed vs. FastSAM

Parámetros ajustables: `MIN_AREA_FRAC` (área mínima de instancia) y `OVERLAP_THRESH`
(fracción de la máscara SAM que debe caer dentro de la región de big rock).


In [ ]:
MIN_AREA_FRAC = 0.0005
OVERLAP_THRESH = 0.5

rows = []
for iid in ids:
    msk = config.MSL_NCAM_LABELS_TRAIN / f"{iid}.png"
    img = mu.mask_to_image_path(msk)
    if img is None:
        continue
    m = mu.read_mask(msk)
    bigrock = mu.big_rock_mask(m)

    ws = int(RESULTS.loc[RESULTS.image_id == iid, "n_rocks"].iloc[0])
    masks = sc.sam_instance_masks(img, model, device=DEVICE)
    sam_n, _ = sc.count_in_region(masks, bigrock, MIN_AREA_FRAC, OVERLAP_THRESH)

    rows.append({"image_id": iid,
                 "coverage": RESULTS.loc[RESULTS.image_id == iid, "rock_coverage_pct"].iloc[0],
                 "watershed": ws, "sam": sam_n, "sam_total_masks": len(masks)})
    print(f"{iid[:34]:<36} watershed={ws:<3} SAM={sam_n:<3} (SAM total={len(masks)})")

comp = pd.DataFrame(rows)
comp.to_csv(root / "outputs" / "comparacion_watershed_sam.csv", index=False)
comp

## 4. Resumen y gráfico


In [ ]:
import matplotlib.pyplot as plt

r = comp[["watershed", "sam"]].corr().iloc[0, 1]
diff = (comp.sam - comp.watershed)
print(f"Correlación watershed–SAM: r = {r:.2f}")
print(f"SAM cuenta en promedio {diff.mean():+.1f} rocas más que el watershed "
      f"(mediana {diff.median():+.0f}).")

fig, ax = plt.subplots(figsize=(6, 6))
lim = max(comp.watershed.max(), comp.sam.max()) + 2
ax.plot([0, lim], [0, lim], "--", color="gray", label="igualdad")
ax.scatter(comp.watershed, comp.sam, c="#c0392b", s=60)
ax.set(xlabel="watershed (clásico)", ylabel="FastSAM (local)",
       title=f"Conteo de rocas por imagen  (r={r:.2f})", xlim=(0, lim), ylim=(0, lim))
ax.legend(); ax.grid(alpha=0.3)
plt.show()

## 5. Visualizar una imagen: máscaras de FastSAM sobre la región de big rock


In [ ]:
iid = comp.sort_values("watershed").iloc[len(comp) // 2].image_id  # una del medio
msk = config.MSL_NCAM_LABELS_TRAIN / f"{iid}.png"
img = mu.mask_to_image_path(msk)
m = mu.read_mask(msk); bigrock = mu.big_rock_mask(m)
from PIL import Image
gray = np.array(Image.open(img).convert("L"))
masks = sc.sam_instance_masks(img, model, device=DEVICE)
_, kept = sc.count_in_region(masks, bigrock, MIN_AREA_FRAC, OVERLAP_THRESH)

fig, ax = plt.subplots(1, 3, figsize=(16, 6))
ax[0].imshow(gray, cmap="gray"); ax[0].set_title("Imagen")
ax[1].imshow(bigrock, cmap="gray"); ax[1].set_title("Región big rock (AI4Mars)")
overlay = np.zeros((*bigrock.shape, 3))
rng = np.random.default_rng(0)
for mk in masks:
    if mk.shape == bigrock.shape and np.logical_and(mk, bigrock).sum() / max(mk.sum(), 1) >= OVERLAP_THRESH \
       and mk.sum() >= MIN_AREA_FRAC * bigrock.size:
        overlay[mk] = rng.random(3)
ax[2].imshow(gray, cmap="gray"); ax[2].imshow(overlay, alpha=0.55)
ax[2].set_title(f"Instancias FastSAM en big rock: {len(kept)}")
for a in ax: a.axis("off")
plt.show()

## 6. Interpretación

- **FastSAM** (foundation model, sin entrenar) segmenta por apariencia visual → tiende a
  **sobresegmentar** (parte una roca en sus texturas internas).
- El **watershed** cuenta bloques sobre la máscara de anotación → conteo más conservador.
- Son **dos nociones de 'roca'** distintas; la comparación (tabla + gráfico) es un resultado
  válido para la discusión (E5) y sostiene el ángulo de aprendizaje automático **sin AWS**.

Ajusta `MIN_AREA_FRAC` y `OVERLAP_THRESH` para hacer el conteo de SAM más o menos estricto.
